# Large and wide tables

Column summaries and integrity checks examine every row. Relationship
discovery screens broadly, then limits follow-up work by budget. This lets
the profiler handle tables with millions of rows and hundreds of columns,
while the amount of relationship evidence depends on the configured
budget.

## What stays exact


In [1]:
import polars as pl
from great_tables import GT, loc, style

_doc_table = pl.DataFrame({'Pass': ['Column cards', 'Duplicate rows', 'Single-column keys', 'Sortedness', 'Dates', 'Arithmetic identities', 'Functional dependencies', 'Order constraints'], 'At any size': ['every row: exact cardinality, null counts, statistics, quantiles, top values, concentration', 'full-row hash plus exact verification', 'exact distinct counts over the full column', 'exact scan of adjacent values', 'exact calendar scan of distinct days', 'fitted on a sample, residuals verified on every pair-complete row', 'screened broadly; reported candidates verified over up to 200,000 rows (`standard`) or 500,000 (`deep`) — full data when the table is smaller', 'every pair screened on a 4,096-row sample of each column; pairs that survive verified on every row']})
(GT(_doc_table)
 .fmt_markdown(columns=['Pass', 'At any size'])
 .tab_header(title='What stays exact')
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


GT(_tbl_data=┌─────────────────────────┬─────────────────────────────────┐
│ Pass                    ┆ At any size                     │
╞═════════════════════════╪═════════════════════════════════╡
│ Column cards            ┆ every row: exact cardinality, … │
│ Duplicate rows          ┆ full-row hash plus exact verif… │
│ Single-column keys      ┆ exact distinct counts over the… │
│ Sortedness              ┆ exact scan of adjacent values   │
│ Dates                   ┆ exact calendar scan of distinc… │
│ Arithmetic identities   ┆ fitted on a sample, residuals … │
│ Functional dependencies ┆ screened broadly; reported can… │
│ Order constraints       ┆ every pair screened on a 4,096… │
└─────────────────────────┴─────────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x109918980>, _boxhead=Boxhead([ColInfo(var='Pass', type=<ColInfoTypeEnum.default: 1>, column_label='Pass', column_align='left', column_width=None, stub_level=None), ColInfo(var='At any size', type=<ColInfoTypeEnum.default: 1>, column_label='At any size', column_align='left', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x1099186e0>, _spanners=Spanners([]), _heading=Heading(title='What stays exact', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x109918d70>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x109920b90>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=1, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=2, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=3, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=4, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=5, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=6, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=7, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='At any size', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='At any size', rownum=1, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='At any size', rownum=2, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='At any size', rownum=3, colnum=None, styles=[CellStyleCss(rule='white-space:

Dense and nullable Int64/UInt64/Float64 columns borrow Arrow buffers directly;
other dtypes can require a normalized copy. This is an in-memory profiler,
so input size and exact statistics still drive memory and runtime.

## Bounded relationship work

Screening is broad and cheap; follow-up is bounded. The budget table in
[](coverage.ipynb) gives the per-tier numbers: at `standard`, the encoded
relationship sample is 20,000 rows over all usable columns, each target is
screened against all columns and fitted against at most 48 shortlisted
predictors over 5,000 rows, all numeric pairs are screened and the 496
strongest get rank statistics, and FD candidates are verified over 200,000
rows. `deep` raises each of those. Coverage reports the limits used for each run.

Here is a wide synthetic table with
relationships planted in the *last* columns — deliberately at the far end
of the column order, because a screener that only looked at the first few
dozen columns would miss them:


In [2]:
from html import escape

from IPython.display import HTML, display

import numpy as np
import polars as pl

import onceover

rng = np.random.default_rng(20241024)
n, width = 100_000, 80

data = {"id": np.arange(n, dtype=np.int64)}
data.update({f"noise_{i:02}": rng.random(n) for i in range(width - 7)})
x = rng.uniform(-1, 1, n)
codes = rng.integers(0, 4, n, dtype=np.int64)
dirty_id = np.arange(n, dtype=np.int64)
dirty_id[-1] = 0
data.update({
    "late_x": x,
    "late_affine": 3 * x + 2,       # arithmetic identity with late_x
    "late_curve": x * x,            # nonlinear response
    "dirty_id": dirty_id,
    "late_code": codes,             # code -> label dependency
})

df = pl.DataFrame(data).with_columns(
    pl.col("late_code").cast(pl.String).alias("late_label")
)
assert df.shape == (n, width)
print(df.shape)

report = onceover.profile(df)
display(HTML(f"<p>{escape(report.overview['headline'])}</p>"))

(100000, 80)


The report recovers the planted affine dependency, nonlinear response,
and code→label lookup in the final columns, and ranks the integer id as
the key candidate. Screening considers every usable column before
shortlisting.

The coverage block records the work performed:

In [3]:
from great_tables import GT, loc, style

rows = [{"pass": p, "status": i["status"], "detail": i["detail"]}
        for p, i in report.coverage.items()]

(GT(pl.DataFrame(rows))
 .tab_header(title=f"Coverage on {df.height:,} rows × {df.width} columns (standard)",
             subtitle="Note the caps: the detectors screened everything, then bounded the deep work")
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"),
            loc.body()))

GT(_tbl_data=┌───────────────────────┬─────────────┬─────────────────────────────────┐
│ pass                  ┆ status      ┆ detail                          │
╞═══════════════════════╪═════════════╪═════════════════════════════════╡
│ arithmetic_identities ┆ ran_exact   ┆ 3 candidate pairs checked, 1 i… │
│ association_pairs     ┆ ran_sampled ┆ 0 nominal pairs, 0 mixed pairs… │
│ association_rules     ┆ ran_sampled ┆ 0 of 0 nominal features (all e… │
│ boundary_mass         ┆ ran_exact   ┆ 79 numeric columns scanned for… │
│ breaks                ┆ skipped     ┆ break scan not run: a default … │
│ …                     ┆ …           ┆ …                               │
│ sentinels             ┆ ran_exact   ┆ 79 numeric columns scanned for… │
│ series                ┆ skipped     ┆ no date or datetime axis to de… │
│ slice_shifts          ┆ skipped     ┆ on-request detector; not in th… │
│ sortedness            ┆ ran_exact   ┆ 1 columns sorted by row order   │
│ target_shapes         ┆ skipped     ┆ runs when a target is designat… │
└───────────────────────┴─────────────┴─────────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x10fc0cfc0>, _boxhead=Boxhead([ColInfo(var='pass', type=<ColInfoTypeEnum.default: 1>, column_label='pass', column_align='left', column_width=None, stub_level=None), ColInfo(var='status', type=<ColInfoTypeEnum.default: 1>, column_label='status', column_align='left', column_width=None, stub_level=None), ColInfo(var='detail', type=<ColInfoTypeEnum.default: 1>, column_label='detail', column_align='left', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10fc45a90>, _spanners=Spanners([]), _heading=Heading(title='Coverage on 100,000 rows × 80 columns (standard)', subtitle='Note the caps: the detectors screened everything, then bounded the deep work', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10fc45f90>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10fc0ce90>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='pass', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='pass', rownum=1, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='pass', rownum=2, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='pass', rownum=3, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='pass', rownum=4, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='pass', rownum=5, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='pass', rownum=6, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='pass', rownum=7, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='pass', rownum=8, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleIn

## The scale benchmark

The scale benchmark checks runtime, resident memory, and recovery together.
Its fixed-seed fixture has 2,000,000 rows and 200 columns, including an
integer primary key, a key with one repeated value, and the three
relationships illustrated above. Those relationships occupy columns beyond
position 64. The script fails unless all eight recovery and cardinality
checks pass.

From the repository root, with the checkout's package installed:

```bash
python -m pip install '.[benchmark]'
python benchmarks/scale.py --rows 2000000 --columns 200 --threads 4 \
  --output /tmp/onceover-scale.json
```

The recorded comparison uses release builds, the standard budget, Python
3.13.7, Polars 2.0.0, and four Rayon threads on a macOS arm64 host with
24 GiB of system memory. These are recorded results, not timings of the
smaller demonstration above.


In [4]:
recorded_scale_results = pl.DataFrame({
    "Measurement": [
        "Profiling wall time", "Peak process RSS", "Increase from pre-profile RSS",
        "Recovery and cardinality checks", "Independent noise targets reported predictable",
    ],
    "0.1.2": ["351.08 s", "5.84 GiB", "2.77 GiB", "8 / 8", "0"],
    "0.1.3": ["79.67 s", "3.47 GiB", "0.40 GiB", "8 / 8", "0"],
})
(GT(recorded_scale_results)
 .tab_header(title="Recorded scale benchmark: 2,000,000 rows × 200 columns",
             subtitle="Standard budget · fixed seed 20241024 · four Rayon threads")
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"),
            loc.body()))


GT(_tbl_data=┌─────────────────────────────────┬──────────┬──────────┐
│ Measurement                     ┆ 0.1.2    ┆ 0.1.3    │
╞═════════════════════════════════╪══════════╪══════════╡
│ Profiling wall time             ┆ 351.08 s ┆ 79.67 s  │
│ Peak process RSS                ┆ 5.84 GiB ┆ 3.47 GiB │
│ Increase from pre-profile RSS   ┆ 2.77 GiB ┆ 0.40 GiB │
│ Recovery and cardinality check… ┆ 8 / 8    ┆ 8 / 8    │
│ Independent noise targets repo… ┆ 0        ┆ 0        │
└─────────────────────────────────┴──────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x10fbfce20>, _boxhead=Boxhead([ColInfo(var='Measurement', type=<ColInfoTypeEnum.default: 1>, column_label='Measurement', column_align='left', column_width=None, stub_level=None), ColInfo(var='0.1.2', type=<ColInfoTypeEnum.default: 1>, column_label='0.1.2', column_align='left', column_width=None, stub_level=None), ColInfo(var='0.1.3', type=<ColInfoTypeEnum.default: 1>, column_label='0.1.3', column_align='left', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10fc46990>, _spanners=Spanners([]), _heading=Heading(title='Recorded scale benchmark: 2,000,000 rows × 200 columns', subtitle='Standard budget · fixed seed 20241024 · four Rayon threads', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10fc0d6e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10fc09c70>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Measurement', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Measurement', rownum=1, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Measurement', rownum=2, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Measurement', rownum=3, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Measurement', rownum=4, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='0.1.2', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='0.1.2', rownum=1, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='0.1.2', rownum=2, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='0.1.2', rownum=3, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='0.1.2', rownum=4, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='0.1.3', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='0.1.3', rownum=1, colnum=None, styles=[CellSty

Both versions recovered all planted structures, with no independent noise
target reported as predictable. RSS includes the input and Python process;
the increase from pre-profile RSS is not an allocation count for scratch
space. The 0.1.2 run briefly shared the host with other validation tasks,
while the 0.1.3 run began after those tasks completed. The comparison is a
single-run observation, not a guarantee of a particular speedup.

On constrained 2-core / 8 GiB executors, post-0.1.7 optimizations to
column extraction, exact counting, quantile sorting, boundary checks, and
shared encoding cut the 200,000 × 200 `standard` fixture from 78.93 s on
four threads (0.1.7) to **17.17 s on two threads** with **1.05 GiB peak
process RSS** (all 9 / 9 checks passed, 0 noise targets).

The [benchmark protocol and records](https://github.com/fcela/onceover/blob/main/docs/benchmarks/large-table-performance.md)
and the [2026-11-22 optimization record](https://github.com/fcela/onceover/blob/main/docs/benchmarks/2026-11-22-architecture-performance-precision.md)
explain the timing boundaries, environment, and interpretation. The older
0.1.1 record predates several default detectors and is not a comparable
runtime baseline.

## What bounded search can miss

Samples can miss rare patterns: a relationship visible in 0.01% of rows may
not appear in a 20,000-row sample. Shortlisting can miss weak signals and
interaction-only signals: a pair of columns that predict a target only
*together* may not survive per-column screening. Capped searches stop
early: beyond 496 pairs (`standard`), the profiler still screens every
numeric pair but computes detailed statistics only for the strongest.

Coverage records sample sizes and search caps, and each finding includes
its validation scope. A run that reports no relationship may still have
missed one.
Increase the budget (`deep`, `sample_rows`, `max_pairs`) or test a
specific hypothesis directly.

```{admonition} Full-data scan costs
:class: note

Bounded *relationship* work does not bound the input: exact column
summaries examine every row, so runtime and memory grow with the table.
The budget tiers bound discovery, not the scan.
```

## Tuning for a big table

The overrides from [](coverage.ipynb) apply directly:

```python
# wider sampling when the standard caps feel tight
report = onceover.profile(df, sample_rows=50_000, max_predictors=96,
                          max_pairs=2_000)

# or a wall-clock budget; coverage lists passes skipped after the deadline
report = onceover.profile(df, deadline_seconds=30.0)
```

Sample and pair overrides change the amount of relationship work. A
deadline can skip later passes, and it stops the dependence screen,
the hotspots scan, and the discontinuities pass mid-search; their
coverage entries state how much of each search was examined. Neither
reduces the rows examined by full-data passes that complete.


## Lazy and file inputs

`profile()` also accepts a Polars LazyFrame or a path to a Parquet,
CSV, or IPC file. The source is collected before profiling; a source
with more rows than the source row cap (default 200,000) is profiled
on a stratified sample — one row per equal stratum of the source,
hash-chosen within the stratum, deterministic for a given seed —
instead of being collected whole. The report's `source` section
records the input kind, the source's full row count, the rows
profiled, and the basis; on a sampled basis every finding's evidence
scope records the sample and the source population — the sample's
row count as `rows_evaluated`, the source's as `population_rows`. `exact=True` collects
the source whole, and `source_row_cap=` / `sample_seed=` set the
cap and the seed.

The 10.12 GiB, 220-column Parquet fixture in
`docs/benchmarks/2026-11-04-source-inputs.md` profiles under a
declared 2 GiB peak-memory budget on this basis: the source sample
is 200,000 rows, and the profile recovers the relationships planted
past column 64.

In [5]:
import polars as pl
import onceover

frame = pl.DataFrame({
    "x": range(60_000),
    "y": [float(i % 97) for i in range(60_000)],
    "label": [f"g{i % 12}" for i in range(60_000)],
})
frame.write_parquet("/tmp/onceover-source-demo.parquet")

by_path = onceover.profile("/tmp/onceover-source-demo.parquet",
                           source_row_cap=10_000)
print(by_path.data["source"])

by_lazy = onceover.profile(frame.lazy(), source_row_cap=10_000)
print(by_lazy.data["source"])

{'kind': 'parquet', 'rows': 60000, 'rows_profiled': 10000, 'basis': 'sampled', 'sample': {'method': 'stratified', 'seed': 0, 'cap': 10000}}
{'kind': 'lazyframe', 'rows': 60000, 'rows_profiled': 10000, 'basis': 'sampled', 'sample': {'method': 'stratified', 'seed': 0, 'cap': 10000}}


Both runs state the same sampled basis: 60,000 source rows,
10,000 rows profiled, and the sample that produced them. A
DataFrame input carries no `source` section — the profile ran on
exactly the frame it was handed.